In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.sparse import csc_array
import scipy.sparse.linalg as spla

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# default parameters of the plot layout
plt.rcParams["text.usetex"] = True  # use latex
plt.rcParams["font.size"] = 13
plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.constrained_layout.use"] = True

from qs_mps.applications.Z2.exact_hamiltonian import *
from qs_mps.sparse_hamiltonians_and_operators import *
from qs_mps.mps_class import MPS
from qs_mps.utils import anim, get_cx, get_cy

In [ ]:
model = "heis"
chi = 30
h_i, h_f = -1.5, 1.5
a = 3e-2
couplings = np.arange(h_i,h_f+a,a)
DMRG2 = False
d = 3
J = 1
eps = 0
Ls = [10]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
model = "heis"
chi = 50
h_i, h_f = -1.5, 1.5
a = 3e-2
couplings = np.arange(h_i,h_f,a)
DMRG2 = False
d = 3
J = 1
eps = 0
Ls = [10,20,30]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
model = "heis"
chi = 100
h_i, h_f = -1.2, -0.8
a = 1e-2
couplings = np.arange(h_i,h_f,a)
DMRG2 = False
d = 3
J = 1
eps = 1e-4
Ls = [10,20]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
model = "heis"
chi = 100
h_i, h_f = -1.2, -0.8
a = 1e-2
couplings = np.arange(h_i,h_f+a,a)
DMRG2 = False
d = 3
J = 1
eps = -1e-2
Ls = [40,60,80]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
model = "heis"
chi = 200
h_i, h_f = 0.8, 1.2
a = 1e-2
couplings = np.arange(h_i,h_f+a,a)
DMRG2 = False
d = 3
J = 1
eps = 1e-3
Ls = [40,60,80]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

## On-axis overlaps

In [ ]:
fidelity = []
for L in Ls:
    fidelity_L = []
    for k in range(len(couplings)-1):
        print(f"L: {L}, coupling: {couplings[k]:.4f}")
        heis_chain_g = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps, J=J, bc='obc')
        heis_chain_g_dg = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k+1], eps=eps, J=J, bc='obc')
        
        heis_chain_g.load_sites(path=path, precision=3)
        heis_chain_g_dg.load_sites(path=path, precision=3)
        heis_chain_g.ancilla_sites = heis_chain_g_dg.sites.copy()

        fidelity_L.append(heis_chain_g._compute_norm(site=1, mixed=True).copy())
    fidelity.append(fidelity_L)
fidelity = np.array(fidelity)

In [ ]:
colors = create_sequential_colors(len(Ls))
offset = [0,0,0]
for i, L in enumerate(Ls):
    dfs_cut, cut = cut_fidelity_susceptibility(fid=fidelity[i,:], offset=offset[i], a=a, discr=True)
    # plt.plot(couplings[cut:-1], dfs_cut, color=colors[i], label=f"$R: {R}$")
    plt.plot(couplings[cut:-1], dfs_cut/L, color=colors[i], label=f"$L: {L}$")

plt.grid(True)
plt.legend()

In [ ]:
colors = create_sequential_colors(len(Ls))

for i, L in enumerate(Ls):
    dfs = discrete_fidelity_susceptibility(fid=fidelity[i,:], a=a)
    # plt.plot(couplings[:-1], dfs, color=colors[i], label=f"$L: {L}$")
    plt.plot(couplings[:-1], dfs/L, color=colors[i], label=f"$L: {L}$")

plt.legend()

## Magnetization

In [ ]:
model = "heis"
chi = 200
h_i, h_f = -1.2, -0.8
a = 1e-2
couplings = np.arange(h_i,h_f+a,a)
DMRG2 = False
d = 3
J = 1
eps = -1e-3
Ls = [40,60,80]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
model = "heis"
chi = 200
h_i, h_f = 0.8, 1.2
a = 1e-2
couplings = np.arange(h_i,h_f+a,a)
DMRG2 = False
d = 3
J = 1
eps = 1e-3
Ls = [40,60,80]
path = "/Users/fradm/Desktop/projects/Fidelities_with_TN"

In [ ]:
mag = []
for L in Ls:
    mag_L = []
    for k in range(len(couplings)):
        print(f"L: {L}, coupling: {couplings[k]:.4f}")
        heis_chain_g = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps, J=J, bc='obc')
        heis_chain_g.load_sites(path=path, precision=3)
        heis_chain_g.mag_3(stag=True)
        # mag_L.append(heis_chain_g.mpo_first_moment().real)
        mag_L.append(heis_chain_g.mpo_second_moment().real)
    mag.append(mag_L)

In [ ]:
colors = create_sequential_colors(len(Ls))

for i, L in enumerate(Ls):
    plt.plot(couplings, mag[i], color=colors[i], label=f'$L= {L}$')
plt.legend()

### Hole states analysis

In [ ]:
hole = np.array([0,1,0])
up = np.array([1,0,0])
down = np.array([0,0,1])
hole = hole.reshape(1,3,1)
up = up.reshape(1,3,1)
down = down.reshape(1,3,1)

In [ ]:
energies_hole_e = []
energies_hole_b = []
energies = []
overlap_gs_hole = []
for L in Ls:
    site = L//2
    boundary_hole_state = [hole] + [up]*(L-1)
    bulk_hole_state = [up]*(site) + [hole] + [up]*(L - 1 - site)

    energies_hole_e_L = []
    energies_hole_b_L = []
    energies_L = []
    overlap_gs_hole_L = []
    for k in range(len(couplings)):
        print(f"L: {L}, coupling: {couplings[k]:.4f}")
        mps = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps, J=J, bc='obc')
        mps.load_sites(path=path, precision=3)
        mps.mpo()
        E_gs = mps.mpo_first_moment().real
        
        mps_a = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps, J=J, bc='obc')
        mps_a.mpo()
        
        mps_a.sites = boundary_hole_state.copy()
        E_hole_e_mps = mps_a.mpo_first_moment().real
        
        mps_a.sites = bulk_hole_state.copy()
        E_hole_b_mps = mps_a.mpo_first_moment().real
        
        mps_a.ancilla_sites = mps.sites.copy()
        overlap_mps = mps_a._compute_norm(site=1, mixed=True).real
        
        energies_hole_e_L.append(E_hole_e_mps)
        energies_hole_b_L.append(E_hole_b_mps)
        energies_L.append(E_gs)
        overlap_gs_hole_L.append(overlap_mps)

    energies_hole_e.append(energies_hole_e_L)
    energies_hole_b.append(energies_hole_b_L)
    energies.append(energies_L)
    overlap_gs_hole.append(overlap_gs_hole_L)

In [ ]:
colors = create_sequential_colors(len(Ls))

for i, L in enumerate(Ls):
    plt.plot(couplings, energies[i], color=colors[i], linewidth=1, label=f"$L={L}$ - $E_0$")
    plt.plot(couplings, energies_hole_e[i], color=colors[i], linestyle='--', linewidth=1, label=f"$L={L}$ - $E_h^e$")
    plt.plot(couplings, energies_hole_b[i], color=colors[i], linestyle='-.', linewidth=1, label=f"$L={L}$ - $E_h^b$")
plt.text(x=-1.2, y=max(energies_hole_e[1])-0.5, s=r"$E_h^{edge} = \langle \psi_h^{edge} | H (J_z) | \psi_h^{edge} \rangle$", fontsize=10)
plt.text(x=-1.2, y=max(energies_hole_e[1])-1.5, s=r"$E_h^{bulk} = \langle \psi_h^{bulk} | H (J_z) | \psi_h^{bulk} \rangle$", fontsize=10)
plt.text(x=-1.2, y=max(energies_hole_e[1])-2.5, s=r"$| \psi_h^{edge} \rangle = | h \uparrow \uparrow \dots \uparrow \rangle$", fontsize=10)
plt.text(x=-1.2, y=max(energies_hole_e[1])-3.5, s=r"$| \psi_h^{bulk} \rangle = | \uparrow \dots \uparrow h \uparrow \uparrow \dots \uparrow \rangle$", fontsize=10)
plt.legend()

In [ ]:
colors = create_sequential_colors(len(Ls))

for i, L in enumerate(Ls):
    plt.plot(couplings, overlap_gs_hole[i], color=colors[i], label=f"$L={L}$")
plt.legend()

In [ ]:
L = 60
h_i = 0.8
h_f = 1.2
a = 1e-2
couplings = np.arange(h_i, h_f+a, a)
npoints = len(couplings)
chi = 200
eps_gs = 1e-3
state = "Neel" # "hole_edge", "hole_bulk", "all_down"

In [ ]:
L = 60
h_i = -1.2
h_f = -0.8
a = 1e-2
couplings = np.arange(h_i, h_f+a, a)
npoints = len(couplings)
chi = 200
eps_gs = -1e-3
state =  "hole_edge" #, "hole_bulk", "all_down", "Neel"
state =  "all_down" #, "hole_bulk", "all_down", "Neel"

In [ ]:
L = 20
h_i = 0
h_f = 10
a = 1
couplings = np.arange(h_i, h_f+a, a)
npoints = len(couplings)
chi = 100
eps_gs = 0
eps_gs = -1e-2
state = "Neel" # "hole_edge", "hole_bulk", "all_down"

In [ ]:
site = L//2
edge_hole_state = [hole] + [up]*(L-1)
down_state = [down]*L
neel_state_up = [down*(i%2) + up*((i+1)%2) for i in range(L)]
neel_state_down = [up*(i%2) + down*((i+1)%2) for i in range(L)]
epss = [1e-1, 1e-2, 1e-3, 1e-4, 0]
# epss = [-1e-1, -1e-2, -1e-3, -1e-4, 0]
energies_state = []
for eps in epss:
    energies_state_eps = []
    for k in range(len(couplings)):
        print(f"L: {L}, coupling: {couplings[k]:.4f}")
        mps_a = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps, J=J, bc='obc')
        mps_a.mpo()
        
        # mps_a.sites = edge_hole_state.copy()
        # mps_a.sites = down_state.copy()
        mps_a.sites = neel_state_up.copy()
        E_state_mps = mps_a.mpo_first_moment().real
        
        energies_state_eps.append(E_state_mps)

    energies_state.append(energies_state_eps)


energies = []
for k in range(len(couplings)):
    print(f"L: {L}, coupling: {couplings[k]:.4f}")
    mps = MPS(L=L, d=d, model=model, chi=chi, h=couplings[k], eps=eps_gs, J=J, bc='obc')
    mps.load_sites(path=path, precision=3)
    mps.mpo()
    E_gs = mps.mpo_first_moment().real
    energies.append(E_gs)


In [ ]:
colors = create_sequential_colors(len(epss))

for i, eps in enumerate(epss):
    ## Neel
    plt.plot(couplings, energies_state[i], color=colors[i], linestyle='--', linewidth=1, label=f"$\epsilon = {eps}$ - $E_N$")
    
    ## Hole
    # plt.plot(couplings, energies_state[i], color=colors[i], linestyle='--', linewidth=1, label=f"$\epsilon = {eps}$ - $E_h^e$")
    
    ## Down
    # plt.plot(couplings, energies_state[i], color=colors[i], linestyle='--', linewidth=1, label=f"$\epsilon = {eps}$ - $E_d$")

plt.plot(couplings, energies, color='r', linewidth=1, label=f"$\epsilon_0 = {eps_gs}$ - $E_0$")

## Neel
plt.text(x=couplings[0], y=min(energies)+1, s=r"$E_N = \langle \psi_{Neel} | H (J_z, \epsilon) | \psi_{Neel} \rangle$", fontsize=10)
plt.text(x=couplings[0], y=min(energies)+2, s=r"$| \psi_{Neel} \rangle = | \downarrow \uparrow \downarrow \dots \downarrow \uparrow \rangle$", fontsize=10)

## Hole
# plt.text(x=couplings[0], y=min(energies)-0.5, s=r"$E_h^e = \langle \psi_{edge} | H (J_z, \epsilon) | \psi_{edge} \rangle$", fontsize=10)
# plt.text(x=couplings[0], y=min(energies)-1, s=r"$| \psi_{edge} \rangle = | h \uparrow \uparrow \dots \uparrow \rangle$", fontsize=10)

## Down
# plt.text(x=couplings[0], y=max(energies)-0.5, s=r"$E_d = \langle \psi_d | H (J_z, \epsilon) | \psi_d \rangle$", fontsize=10)
# plt.text(x=couplings[0], y=max(energies)-1, s=r"$| \psi_d \rangle = | \downarrow \downarrow \dots \downarrow \rangle$", fontsize=10)

plt.title(f"$L={L}$")
plt.xlabel(r"$J_z$")
plt.legend()
plt.savefig(f"{path}/figures/energy_gs_vs_{state}_L_{L}_h_{h_i}-{h_f}_npoints_{npoints}_eps_{eps_gs}_diff_eps_chi_{chi}.pdf", dpi=300, format="pdf")

In [ ]:
plt.plot(couplings, energies_state[2], color='k', marker='1', linewidth=0.5, label=f"$\epsilon = {epss[2]}$ - $E_d$")
plt.plot(couplings, energies, color='r', linewidth=0.8, label=f"$\epsilon_0 = {eps_gs}$ - $E_0$")
plt.text(x=-1.2, y=max(energies_state[2])-2, s=r"$E_d = \langle \psi_d | H (J_z, \epsilon) | \psi_d \rangle$", fontsize=12)
plt.text(x=-1.2, y=max(energies_state[2])-2.5, s=r"$| \psi_d \rangle = | \downarrow \downarrow \dots \downarrow \rangle$", fontsize=12)
plt.title(f"$L={L}$")
plt.xlabel(r"$J_z$")
plt.legend()
plt.savefig(f"{path}/figures/energy_gs_vs_all_down_L_{L}_h_{h_i}-{h_f}_npoints_{npoints}_eps_{eps_gs}_chi_{chi}.pdf", dpi=300, format="pdf")

## Off-axis overlaps

In [ ]:
fidelity_off_axis = []
for R in Rs:
    fidelity_R = []
    for k in range(len(couplings)-1):
        print(f"R: {R}, coupling: {couplings[k]:.4f}")
        z2_lattice_g = MPS(L=L, d=2**N, model=model, chi=chi, h=couplings[k], bc='pbc')
        z2_lattice_g.Z2.add_charges(rows=get_cx(L=L,R=R),columns=[0,1])
        z2_lattice_g.charges = z2_lattice_g.Z2.charges
        z2_lattice_g.Z2._define_sector()
        z2_lattice_g_dg = MPS(L=L, d=2**N, model=model, chi=chi, h=couplings[k+1], bc='pbc')
        z2_lattice_g_dg.Z2.add_charges(rows=get_cx(L=L,R=R),columns=[0,1])
        z2_lattice_g_dg.charges = z2_lattice_g_dg.Z2.charges
        z2_lattice_g_dg.Z2._define_sector()
        
        z2_lattice_g.load_sites(path=path, precision=3, cx=get_cx(L=L,R=R),cy=[0,1])
        z2_lattice_g_dg.load_sites(path=path, precision=3, cx=get_cx(L=L,R=R),cy=[0,1])
        z2_lattice_g.ancilla_sites = z2_lattice_g_dg.sites.copy()

        fidelity_R.append(z2_lattice_g._compute_norm(site=1, mixed=True).copy())
    fidelity_off_axis.append(fidelity_R)
fidelity_off_axis = np.array(fidelity_off_axis)

In [ ]:
colors = create_sequential_colors(len(Rs))

for i, R in enumerate(Rs):
    dfs = discrete_fidelity_susceptibility(fid=fidelity_off_axis[i,:], a=a)
    # plt.plot(couplings[:-1], dfs, color=colors[i], label=f"$R: {R}$")
    plt.plot(couplings[:-1], dfs/R, color=colors[i], label=f"$R: {R}$")

plt.legend()

## Compare on-axis overlaps with off-axis overlaps

In [ ]:
for i, R in enumerate(Rs):
    dfs = discrete_fidelity_susceptibility(fid=fidelity[i,:], a=a)
    dfs_off = discrete_fidelity_susceptibility(fid=fidelity_off_axis[i,:], a=a)
    # plt.plot(couplings[:-1], dfs, color=colors[i], label=f"$R: {R}$")
    plt.plot(couplings[:-1], dfs/R, color=colors[i], label=f"$R: {R}$")
    plt.plot(couplings[:-1], dfs_off/R, color=colors[i], linestyle='--', label=f"$R off: {R}$")

plt.legend()

In [ ]:
from qs_mps.applications.Z2.relevant_observables import fit_params_sys

def pot_diff_plots(l, L, chis, bc, sector, h_i, h_f, npoints, path, param, fit):
    euclidean = True
    manhatten = False
    
    cx = None
    cy = None
    Rs = [25,26,27,28,29,30]
    sigmas_on, sigmas_on_err, sigmas_on_ris, sigmas_on_ris_err, list_ris = fit_params_sys(Rs, l, L, chis, bc, sector, h_i, h_f, npoints, path, cx, cy, param, fit, ris=True)

    cx = None
    cy = None
    Rs = [25,26,27,28,29,30]
    sigmas_off, sigmas_off_err, sigmas_off_ris, sigmas_off_ris_err, list_ris = fit_params_sys(Rs, l, L, chis, bc, sector, h_i, h_f, npoints, path, cx, cy, param, fit, euclidean=euclidean, manhatten=manhatten, ris=True)

    obs = np.array(sigmas_off) - np.array(sigmas_on)
    obs_err = np.array(sigmas_off_err) + np.array(sigmas_on_err)

    return obs, obs_err

def rri_plot(ax):
    ax.grid(color="gray", linestyle=":")
    ax.patch.set_facecolor('none')
    
    colors = ["#ff0f7b", "#f89b29"]
    colors = ["#4A8FE7", "#D95D39"]
    colors = ["firebrick", "teal"]

    l = 5
    L = 50
    fit = 1

    left_axis_limits = (-0.5e-3,1e-3)
    right_axis_limits = (0.5,5)

    chis = [64,128]
    bc = 'pbc'
    sector = "2_particle(s)_sector"

    param = 0

    if param == 0:
        label='$\\sigma_{\\mathrm{off}} - \\sigma_{\\mathrm{on}}$'
    elif param == 1:
        label="$\\gamma_{\\mathrm{off}} - \\gamma_{\\mathrm{on}}$"
    elif param == -1:
        label="$\\beta_{\\mathrm{off}} - \\beta_{\\mathrm{on}}$"
    
    rescale = 1
    h_i, h_f, npoints = 0.8, 1.1, 31
    gs = np.linspace(h_i,h_f,npoints)
    obs, obs_err = pot_diff_plots(l, L, chis, bc, sector, h_i, h_f, npoints, path, param, fit=fit)
    top_plot, = ax.plot(gs[0::2], rescale*obs[0::2], markersize=5, color=colors[0], marker='d', fillstyle=None, mew=0.5, mfc='w', label=label, zorder=5)
    ax.fill_between(x=gs[0::2], y1=rescale*(obs[0::2]-obs_err[0::2]), y2=rescale*(obs[0::2]+obs_err[0::2]), color=colors[0], alpha=0.4, zorder=4)


    # h_i, h_f, npoints = 1.0, 2.0, 6
    # gs = np.linspace(h_i,h_f,npoints)
    # label = None
    # obs, obs_err = pot_diff_plots(l, L, chis, bc, sector, h_i, h_f, npoints, path, param, fit=fit)
    # ax.plot(gs[0:], rescale*obs[0:], markersize=5, color=colors[0], marker='d', fillstyle=None, mew=0.5, mfc='w', label=label, zorder=3)
    # ax.fill_between(x=gs[0:], y1=rescale*(obs[0:]-obs_err[0:]), y2=rescale*(obs[0:]+obs_err[0:]), color=colors[0], alpha=0.4, zorder=2)
    
    # Create a second y-axis sharing the same x-axis
    ax2 = ax.twinx()

    ax2.set_zorder(ax.get_zorder() - 1)
    # ax2.patch.set_visible(False)

    param = -1

    if param == 0:
        label=r'$\sigma{\mathrm{off}} - \sigma{\mathrm{on}}$'
    elif param == 1:
        label=r"$\\gamma_{\\mathrm{off}} - \\gamma_{\\mathrm{on}}$"
    elif param == -1:
        label='$\\beta_{\\mathrm{off}} - \\beta_{\\mathrm{on}}$'
    
    h_i, h_f, npoints = 0.8, 1.1, 31
    colors = create_sequential_colors(len(Rs))
    for i, R in enumerate(Rs):
        dfs = discrete_fidelity_susceptibility(fid=fidelity[i,:], a=a)
        dfs_off = discrete_fidelity_susceptibility(fid=fidelity_off_axis[i,:], a=a)
        # plt.plot(couplings[:-1], dfs, color=colors[i], label=f"$R: {R}$")
        ax2.plot(couplings[:-1], dfs/R, color=colors[i])
        ax2.plot(couplings[:-1], dfs_off/R, color=colors[i], linestyle='--')

    # h_i, h_f, npoints = 1.0, 2.0, 6
    # gs = np.linspace(h_i,h_f,npoints)
    # label = None
    # obs, obs_err = pot_diff_plots(l, L, chis, bc, sector, h_i, h_f, npoints, path, param, fit=fit)
    # ax2.plot(gs[0:], obs[0:], markersize=7, color=colors[1], marker='P', fillstyle=None, mew=0.5, mfc='w', label=label, zorder=1)
    # ax2.fill_between(x=gs[0:], y1=obs[0:]-obs_err[0:], y2=obs[0:]+obs_err[0:], color=colors[1], alpha=0.4, zorder=0)


    # Force exponential notation on the x-axis
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((-1, 1))  # Force scientific notation outside this range
    formatter.set_useOffset(False)

    gr = 0.91
    ax2.axvspan(0.79, gr, color="#009B72", alpha=0.10, zorder=0)
    ax2.vlines(gr, right_axis_limits[0], right_axis_limits[1]+right_axis_limits[1]*0.05, 'k', '--', linewidth=0.8, zorder=1)
    ax2.axvspan(gr, 1.11, color="#009DDC", alpha=0.10, zorder=0)
    ax2.text(gr+0.01, right_axis_limits[0]+0.18, '$g_{\\mathrm{r}}$')
    
    
    ax.yaxis.set_major_formatter(formatter)
    ax.ticklabel_format(style='sci', axis='y')
    ax.set_ylabel("$\\sigma_{\\mathrm{off}} - \\sigma_{\\mathrm{on}}$", color=colors[0])
    ax.tick_params(axis='y', labelcolor=colors[0])

    ax2.yaxis.set_major_formatter(formatter)
    ax2.ticklabel_format(style='sci', axis='y')
    ax2.set_ylabel("$\\chi_{\\mathcal{F}} = \\frac{2}{(\delta g)^2}\\left( 1 - \\langle \\psi(g)| \\psi(g+ \delta g)\\rangle \\right)$", color=colors[1])
    ax2.tick_params(axis='y', labelcolor=colors[1])

    ax2.set_ylim((right_axis_limits[0], right_axis_limits[1]+right_axis_limits[1]*0.05))
    ax.set_ylim((left_axis_limits[0], left_axis_limits[1]+left_axis_limits[1]*0.05))
    ax.set_xlim(left=0.79, right=1.11)
    ax2.set_xlim(left=0.79, right=1.11)
    ax.set_yticks(ticks=np.linspace(left_axis_limits[0],left_axis_limits[1],4), labels=["$-0.0005$","$0$","$0.0005$","$0.0010$"])
    ax.set_xticks(ticks=[0.8,0.9,1.0,1.1], labels=[0.8,0.9,1.0,1.1])
    ax2.set_yticks(ticks=np.linspace(0.5,4.5,5), labels=["$0.5$","$1.5$","$2.5$","$3.5$","$4.5$"])

    # ax.set_xticks(ticks=[0.4,0.6,0.8,1.0], labels=[0.4,0.6,0.8,1.0])
    # ax.set_yticks(ticks=[0,0.5,1,1.5], labels=[0.,0.5,1,1.5])
    
    # --- proxy legend handles ---
    solid_line = Line2D(
        [], [], color='black', linestyle='-', linewidth=2,
        label='on-axis'
    )

    dashed_line = Line2D(
        [], [], color='black', linestyle='--', linewidth=2,
        label='off -axis'
    )



    handles1, labels1 = ax.get_legend_handles_labels()
    # handles2, labels2 = ax2.get_legend_handles_labels()
    
    # leg1 = ax.legend(handles1, labels1, loc='upper left')
    # ax.add_artist(leg1)
    ax2.text(0.95, 4.5, f"$R={Rs}$")
    ax.legend(handles=[top_plot, solid_line, dashed_line], loc='upper left')
    # Merge and show the legend only in ax1
    ax.set_xlabel(r"$g$")
    ax.set_title("(c)", loc='left', fontsize=14)


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8,8))

rri_plot(ax)

Taking the difference of on-axis and off-axis fidelity susceptibilities would correspond in  
how differently the two string configurations respond to the same microscopic deformation in the parameter space (coupling change $\delta g$).  
The on-axis undergoes the roughening transition, while the off-axis string is already rough.
Then, we expect a large difference happening at the roughening point.

In [ ]:
for i, R in enumerate(Rs):
    dfs = discrete_fidelity_susceptibility(fid=fidelity[i,:], a=a)
    dfs_off = discrete_fidelity_susceptibility(fid=fidelity_off_axis[i,:], a=a)
    # plt.plot(couplings[:-1], dfs, color=colors[i], label=f"$R: {R}$")
    plt.plot(couplings[:-1], dfs/R - dfs_off/R, color=colors[i], label=f"$R: {R}$")

plt.legend()

## Overlap between different sectors $\langle \psi_{on}(g) | \psi_{off}(g) \rangle$

In [ ]:
fidelity_on_vs_off = []
for R in Rs:
    fidelity_R = []
    for k in range(len(couplings)):
        print(f"R: {R}, coupling: {couplings[k]:.4f}")
        z2_lattice_on = MPS(L=L, d=2**N, model=model, chi=chi, h=couplings[k], bc='pbc')
        z2_lattice_on.Z2.add_charges(rows=get_cx(L=L,R=R),columns=[0,0])
        z2_lattice_on.charges = z2_lattice_on.Z2.charges
        z2_lattice_on.Z2._define_sector()
        z2_lattice_off = MPS(L=L, d=2**N, model=model, chi=chi, h=couplings[k], bc='pbc')
        z2_lattice_off.Z2.add_charges(rows=get_cx(L=L,R=R),columns=[0,1])
        z2_lattice_off.charges = z2_lattice_off.Z2.charges
        z2_lattice_off.Z2._define_sector()
        
        z2_lattice_on.load_sites(path=path, precision=3, cx=get_cx(L=L,R=R),cy=[0,0])
        z2_lattice_off.load_sites(path=path, precision=3, cx=get_cx(L=L,R=R),cy=[0,1])
        z2_lattice_on.ancilla_sites = z2_lattice_off.sites.copy()

        fidelity_R.append(z2_lattice_on._compute_norm(site=1, mixed=True).copy())
    fidelity_on_vs_off.append(fidelity_R)
fidelity_on_vs_off = np.array(fidelity_on_vs_off)

In [ ]:
for i, R in enumerate(Rs):
    plt.plot(couplings, np.abs(fidelity_on_vs_off[i,:]), color=colors[i], label=f"$R: {R}$")
plt.legend()

This quantity should be zero, independently of the coupling. If the two string configurations, for the limit $R \to \infty$, are equivalent but for the angle, in the roughening phase isotropy would give two degenerate states. The two states, constitute part of the basis relative to the energy eigenvalue for a certain coupling within the rough phase. Hence, the states are orthogonal.

## Data collapse

In [ ]:
colors = create_sequential_colors(len(Rs))
offset = [10,0,0,0,0,0,0,0,0,0]
y_max = []
x_max = []
for i, R in enumerate(Rs):
    dfs_cut, cut = cut_fidelity_susceptibility(fid=fidelity[i,:], offset=offset[i], a=a, discr=True)
    y_max.append(np.max(dfs_cut))
    x_max.append(couplings[cut+np.argmax(dfs_cut)])

In [ ]:
x_max, y_max

In [ ]:
def linear_law(x, a, b):
    return a * x + b

def power_law(x, a, b, c):
    return a * (x ** b) + c

## using maxima peaks to extract the value of nu

popt, cov = curve_fit(linear_law, np.log(Rs)[0:], np.log(y_max)[0:])

a_fit = popt[0]
a_err = np.sqrt(np.diag(cov))[0]

# Calculate nu and its error
nu_fit = 2 / a_fit
nu_err = (2 / a_fit**2) * a_err


## using the pseudocritical points to extract the value of g_c

popt, cov = curve_fit(power_law, [1/R for R in Rs][0:], x_max[0:])

g_c_fit = popt[2]
g_c_err = np.sqrt(np.diag(cov))[2]

print(f"nu: {nu_fit}+/-{nu_err}")
print(f"g_c: {g_c_fit}+/-{g_c_err}")

In [ ]:

colors = create_sequential_colors(len(Rs))
for i, R in enumerate(Rs):
    dfs_cut, cut = cut_fidelity_susceptibility(fid=fidelity[i,:], offset=offset[i], a=a, discr=True)
    plt.plot((couplings[cut:-1]-g_c_fit)*(R**(1/nu_fit)), dfs_cut/(R**(2/nu_fit)), color=colors[i], linestyle='-', label=f"$R: {R}$")


plt.xlabel("$R^{(1/\\nu)} (g-g_c)$")
plt.ylabel("$R^{(-2/\\nu)} \\chi_F$")
plt.grid(True)
plt.legend()